# Forecasting: SBA vs a time-series foundation model

Self-contained. Generates its own data, defines its own models, needs no other file
in this repository.

**The question.** Does a time-series foundation model beat classical intermittent-demand
methods on warung SKUs? The literature review (`docs/run-a-research.html`) predicts *no* —
classical wins on short, sparse, lumpy series. This tests that.

**Two things get measured, and they can disagree:**

1. Forecast error (MASE / RMSSE) — the usual answer.
2. Inventory outcome — each forecast drives a reorder policy; count stockouts and stock
   held. This is what the shop feels, and it rewards *calibrated uncertainty*, not just
   an accurate point estimate.

A method can lose on (1) and win on (2). That asymmetry is the interesting result.


## Choose a GPU

This box is shared. Pick a card that is idle, and mask the others so a stray
`.to('cuda')` cannot land on somebody else's training run.


In [ ]:
# ── GPU selection ────────────────────────────────────────────────────────────
# Set this BEFORE torch or paddle is imported. Once either initialises CUDA the
# variable is ignored — if you have already run the imports, restart the kernel.
import os, subprocess

try:
    print(subprocess.run(
        ['nvidia-smi', '--query-gpu=index,name,utilization.gpu,memory.used,memory.total',
         '--format=csv'], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print('nvidia-smi not found — no NVIDIA GPU visible on this machine.')

GPU = 1          # which physical GPU to use.  None = leave all visible.

if GPU is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU)
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', '<unset: all GPUs>'))
print('Inside this notebook that GPU is addressed as device 0.')

In [ ]:
# !pip install pandas numpy matplotlib statsforecast chronos-forecasting torch

import math, collections, warnings
from datetime import date, timedelta
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SEED, DAYS, HORIZON, HOLDOUT, LEAD = 42, 180, 14, 28, 3

# Fixed categorical order, assigned per method, never cycled.
PALETTE = {'SeasonalNaive':'#2a78d6', 'AutoETS':'#eb6834', 'CrostonSBA':'#1baf7a',
           'TSB':'#eda100', 'Chronos':'#e87ba4'}
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.grid':True, 'grid.alpha':.25, 'font.size':9})

## 1 · The catalogue

200 products with real names and prices, recovered from the nota-pembelian corpus by
column geometry. Each SKU's archetype comes from how often it actually appears in
receipts — products warungs genuinely restock often become fast movers.

`p_sell` is the chance the SKU moves at all on a given day; `r, p` shape the negative
binomial that decides quantity when it does.


In [ ]:
CATALOGUE = [
    ('es teh', 5000, 'fast'),  ('es jeruk', 7000, 'fast'),
    ('teh hangat', 5000, 'fast'),  ('air es', 2000, 'fast'),
    ('pulpen', 5000, 'fast'),  ('garam', 6000, 'fast'),
    ('gula', 17000, 'fast'),  ('air putih', 2000, 'fast'),
    ('nasi kotak', 25000, 'fast'),  ('snack', 15000, 'fast'),
    ('gorengan', 2000, 'fast'),  ('es campur', 15000, 'fast'),
    ('es kelapa', 7000, 'fast'),  ('hvs a 4 sidu', 55000, 'fast'),
    ('gula pasir', 13000, 'fast'),  ('mie goreng', 15000, 'fast'),
    ('minyak goreng', 16000, 'fast'),  ('nasi goreng', 15000, 'fast'),
    ('penghapus', 4000, 'fast'),  ('telur', 2500, 'fast'),
    ('tepung', 18000, 'steady'),  ('buku gambar', 12000, 'steady'),
    ('kopi', 2000, 'steady'),  ('lemper', 2500, 'steady'),
    ('risoles', 2500, 'steady'),  ('lontong', 4000, 'steady'),
    ('peraut', 2000, 'steady'),  ('soto ayam', 18000, 'steady'),
    ('stapler', 15000, 'steady'),  ('teh', 5500, 'steady'),
    ('tepung terigu', 12000, 'steady'),  ('tipe x', 7000, 'steady'),
    ('arem arem', 2500, 'steady'),  ('blue band', 8000, 'steady'),
    ('bolu kukus', 2000, 'steady'),  ('brosur', 2000, 'steady'),
    ('charger hp', 60000, 'steady'),  ('nastar 1 kg', 68000, 'steady'),
    ('penggaris', 5000, 'steady'),  ('pensil', 4500, 'steady'),
    ('putri salju 1 kg', 65000, 'steady'),  ('putu ayu', 2500, 'steady'),
    ('semprit 1 kg', 60000, 'steady'),  ('cucur', 2000, 'steady'),
    ('cutter', 7000, 'steady'),  ('earphone', 45000, 'steady'),
    ('ganti baterai', 150000, 'steady'),  ('hvs a 3 sidu', 96000, 'steady'),
    ('indomie', 3000, 'steady'),  ('nasi sop', 17000, 'steady'),
    ('sate ayam', 2000, 'steady'),  ('spanduk', 35000, 'steady'),
    ('sticky note', 8000, 'steady'),  ('teh celup', 6000, 'steady'),
    ('telur ayam', 2000, 'steady'),  ('buku tulis pack', 48000, 'steady'),
    ('es teh tawar', 3000, 'steady'),  ('gunting', 8000, 'steady'),
    ('kentang balado', 17000, 'steady'),  ('kertas hvs a 4', 45000, 'steady'),
    ('kopi sachet', 2000, 'intermittent'),  ('lakban bening', 15000, 'intermittent'),
    ('margarin', 8000, 'intermittent'),  ('nasi mawut', 15000, 'intermittent'),
    ('pisang goreng', 2000, 'intermittent'),  ('pulpen pack', 24000, 'intermittent'),
    ('rawon', 22000, 'intermittent'),  ('bakaran paha', 6000, 'intermittent'),
    ('bakwan', 2000, 'intermittent'),  ('bubur ayam', 15000, 'intermittent'),
    ('es durian semok', 28000, 'intermittent'),  ('es susu putih', 7000, 'intermittent'),
    ('es teler', 25000, 'intermittent'),  ('ganti kabel', 25000, 'intermittent'),
    ('indomie ayam', 21000, 'intermittent'),  ('isolasi', 17000, 'intermittent'),
    ('kastengel 1 kg', 130000, 'intermittent'),  ('kerupuk', 3000, 'intermittent'),
    ('lem fox', 15000, 'intermittent'),  ('mie kari ayam', 3000, 'intermittent'),
    ('mie soto banjar', 3500, 'intermittent'),  ('minyak goreng 1 l', 17000, 'intermittent'),
    ('mouse', 50000, 'intermittent'),  ('nasi pecel', 25000, 'intermittent'),
    ('nota kontan', 6000, 'intermittent'),  ('paha bakar', 8000, 'intermittent'),
    ('pentol bakar', 5000, 'intermittent'),  ('pisang keju', 18000, 'intermittent'),
    ('roti bakar coklat', 17000, 'intermittent'),  ('roti keju', 120000, 'intermittent'),
    ('roti nanas', 11000, 'intermittent'),  ('sate bebek', 2500, 'intermittent'),
    ('servis kipas angin', 50000, 'intermittent'),  ('sosis bakar', 10000, 'intermittent'),
    ('steropom', 10000, 'intermittent'),  ('stop map', 2000, 'intermittent'),
    ('tahu isi', 2000, 'intermittent'),  ('telur bebek', 3000, 'intermittent'),
    ('telur puyuh', 500, 'intermittent'),  ('air mineral', 2000, 'intermittent'),
    ('anti gores', 20000, 'intermittent'),  ('ayam gami teri', 36000, 'intermittent'),
    ('ayam sambal gami', 36000, 'intermittent'),  ('ayam sambal korek', 23000, 'intermittent'),
    ('bakaran kulit', 3000, 'intermittent'),  ('bakaran sayap', 7000, 'intermittent'),
    ('bakaran usus', 4000, 'intermittent'),  ('bakso', 20000, 'intermittent'),
    ('bebek lalapan', 48000, 'intermittent'),  ('buku tulis', 45000, 'intermittent'),
    ('capcay kuah ayam', 23000, 'intermittent'),  ('casing hp', 35000, 'intermittent'),
    ('cetak foto 30 r', 150000, 'intermittent'),  ('cuci motor', 10000, 'intermittent'),
    ('deterjen', 15000, 'intermittent'),  ('donat', 2000, 'intermittent'),
    ('es buah', 23000, 'intermittent'),  ('es campur durian', 35000, 'intermittent'),
    ('es oyen durian', 28000, 'intermittent'),  ('es susu coklat', 8000, 'intermittent'),
    ('es teler durian', 35000, 'lumpy'),  ('extra joss susu', 11000, 'lumpy'),
    ('gado gado', 17000, 'lumpy'),  ('ganti busi', 15000, 'lumpy'),
    ('ganti oli motor', 45000, 'lumpy'),  ('gorengan pisang', 2000, 'lumpy'),
    ('hvs a 3', 500, 'lumpy'),  ('jasa pengecekan', 20000, 'lumpy'),
    ('jeruk hangat', 7000, 'lumpy'),  ('jus mangga', 17000, 'lumpy'),
    ('kabel hdmi', 45000, 'lumpy'),  ('kacang telor 1 kg', 30000, 'lumpy'),
    ('kertas folio', 25000, 'lumpy'),  ('kopi hitam', 7000, 'lumpy'),
    ('kue cincin', 1500, 'lumpy'),  ('kulit bakar', 3000, 'lumpy'),
    ('lakban hitam', 13000, 'lumpy'),  ('lidah kucing 1 kg', 125000, 'lumpy'),
    ('map plastik', 5000, 'lumpy'),  ('mentega', 18000, 'lumpy'),
    ('mie bancir telor', 21000, 'lumpy'),  ('mie rebus', 8000, 'lumpy'),
    ('nasi ayam kremes', 25000, 'lumpy'),  ('nasi bakar ayam', 12000, 'lumpy'),
    ('nasi bakar teri', 12000, 'lumpy'),  ('nasi campur', 20000, 'lumpy'),
    ('nasi goreng ayam', 26000, 'lumpy'),  ('nasi sop ceker', 30000, 'lumpy'),
    ('pas foto 2 x 3', 1000, 'lumpy'),  ('pas foto 4 x 6', 1500, 'lumpy'),
    ('patin sambal gami', 36000, 'lumpy'),  ('pop mie', 10000, 'lumpy'),
    ('roti abon', 11000, 'lumpy'),  ('roti bakar keju', 17000, 'lumpy'),
    ('roti coklat', 10000, 'lumpy'),  ('sayap bakar', 7000, 'lumpy'),
    ('semen', 65000, 'lumpy'),  ('sempol ayam', 2000, 'lumpy'),
    ('sempol kulit', 2000, 'lumpy'),  ('servis tombol power', 150000, 'lumpy'),
    ('sirup botol', 18000, 'lumpy'),  ('soto banjar', 20000, 'lumpy'),
    ('sterofom', 12000, 'lumpy'),  ('susu kental manis', 13000, 'lumpy'),
    ('tahu bacem', 2000, 'lumpy'),  ('tahu penyet', 11000, 'lumpy'),
    ('teh kotak dus', 66000, 'lumpy'),  ('tela tela bbq', 17000, 'lumpy'),
    ('telor puyuh', 5000, 'lumpy'),  ('usus bakar', 3000, 'lumpy'),
    ('1 kg', 55000, 'slow'),  ('300', 25000, 'slow'),
    ('air mineral botol', 3000, 'slow'),  ('air mineral dus', 22000, 'slow'),
    ('air mineral proof', 24000, 'slow'),  ('ayam asam manis', 30000, 'slow'),
    ('ayam dabu dabu', 36000, 'slow'),  ('ayam geprek korek', 25000, 'slow'),
    ('ayam geprek pete', 23000, 'slow'),  ('ayam geprek teri', 23000, 'slow'),
    ('ayam kremes lalapan', 22000, 'slow'),  ('ayam lalapan', 22000, 'slow'),
    ('ayam sambal matah', 23000, 'slow'),  ('baju lengan panjang', 125000, 'slow'),
    ('baju lengan pendek', 120000, 'slow'),  ('bakar', 3000, 'slow'),
    ('bakaran leher', 5000, 'slow'),  ('baking', 10000, 'slow'),
    ('bancir biasa', 18000, 'slow'),  ('baskom', 8000, 'slow'),
    ('batu asahan', 5000, 'slow'),  ('bawal sambal gami', 40000, 'slow'),
    ('bawan', 2000, 'slow'),  ('bebek sambal gami', 48000, 'slow'),
    ('beli bantal', 45000, 'slow'),  ('benang jahit', 2500, 'slow'),
    ('beras lahap 5 kg', 68000, 'slow'),  ('beras lele 5 kg', 85000, 'slow'),
    ('beras mangkok 10 kg', 121000, 'slow'),  ('beras mangkok 5', 82000, 'slow'),
]

ARCHETYPES = {
    'fast':         dict(p_sell=0.97, r=9.0, p=0.55),
    'steady':       dict(p_sell=0.78, r=4.0, p=0.55),
    'intermittent': dict(p_sell=0.34, r=2.5, p=0.60),
    'lumpy':        dict(p_sell=0.22, r=1.1, p=0.28),
    'slow':         dict(p_sell=0.07, r=1.5, p=0.70),
}

WEEKDAY_MULT = [1.00, 1.00, 1.02, 1.05, 1.15, 1.35, 1.28]   # Mon..Sun

def payday_mult(day_of_month):
    """Indonesian gajian clusters at month end and the first days after."""
    return 1.30 if (day_of_month >= 25 or day_of_month <= 2) else 1.0

print(f'{len(CATALOGUE)} SKUs')
print(collections.Counter(a for _, _, a in CATALOGUE))

## 2 · Generate demand

Two draws per SKU per day: *does it sell*, then *how many*. Seasonality multiplies both.

Deliberately **not** drawn from Croston's own assumptions — the seasonality makes the
occurrence probability time-varying, which Croston does not assume. Generating from a
model and then fitting that same model would make the comparison circular.


In [ ]:
def generate_demand(days=DAYS, seed=SEED, catalogue=CATALOGUE, start=None):
    rng = np.random.default_rng(seed)
    start = start or date.today() - timedelta(days=days)
    rows = []
    for sku_id, (name, price, archetype) in enumerate(catalogue, start=1):
        cfg = ARCHETYPES[archetype]
        for offset in range(days):
            day = start + timedelta(days=offset)
            season = WEEKDAY_MULT[day.weekday()] * payday_mult(day.day)
            if rng.random() > min(cfg['p_sell'] * season, 0.99):
                continue
            qty = int(round(rng.negative_binomial(cfg['r'], cfg['p']) * season))
            if qty <= 0:
                continue
            rows.append({'sale_date': day.isoformat(), 'sku_id': sku_id,
                         'product_name': name, 'qty': qty, 'unit_price': price,
                         'archetype': archetype})
    return rows

demand_rows = generate_demand()
print(f'{len(demand_rows):,} demand lines over {DAYS} days')
pd.DataFrame(demand_rows).head(5)

## 3 · Simulate the shop

Demand is not what a shop records. A shop records what it *sold* — capped by what was on
the shelf — plus what it bought from distributors. Those purchases are exactly the nota
line items the OCR pipeline ingests.

A naive shopkeeper reorders on a fixed rule with a supplier lead time. Its stockout rate
and average holding are the numbers SNAPTOCK's recommendation has to beat.


In [ ]:
def simulate_shop(demand_rows, seed=SEED, catalogue=CATALOGUE, review_days=21):
    rng = np.random.default_rng(seed)
    demand = collections.defaultdict(dict)
    for r in demand_rows:
        demand[r['sku_id']][r['sale_date']] = r['qty']

    days = sorted({r['sale_date'] for r in demand_rows})
    start, end = date.fromisoformat(days[0]), date.fromisoformat(days[-1])
    span = [start + timedelta(days=i) for i in range((end - start).days + 1)]

    purchases, ledger = [], []
    for sku_id, (name, price, archetype) in enumerate(catalogue, start=1):
        series = demand.get(sku_id, {})
        mean_daily = sum(series.values()) / max(len(span), 1)
        lead = int(rng.integers(1, 5))
        reorder = max(1, math.ceil(mean_daily * (lead + 4)))
        order_up_to = max(2, math.ceil(mean_daily * review_days))
        on_hand, incoming = order_up_to, collections.Counter()

        for day in span:
            iso = day.isoformat()
            received = incoming.pop(day, 0)
            on_hand += received
            if received:
                purchases.append({'purchase_date': iso, 'sku_id': sku_id,
                                  'product_name': name, 'qty': received,
                                  'unit_price': price, 'total': received * price})
            want = series.get(iso, 0)
            sold = min(want, on_hand)
            opening = on_hand
            on_hand -= sold
            if on_hand <= reorder and not incoming:
                incoming[day + timedelta(days=lead)] += max(1, order_up_to - on_hand)
            ledger.append({'date': iso, 'sku_id': sku_id, 'product_name': name,
                           'archetype': archetype, 'opening': opening, 'demand': want,
                           'sold': sold, 'lost': want - sold, 'received': received,
                           'closing': on_hand})
    return purchases, ledger

purchases, ledger = simulate_shop(demand_rows)
led = pd.DataFrame(ledger)
print(f'{len(purchases):,} purchases · {len(led):,} ledger rows')
print()
print('purchases.head() — these have the same shape as a nota line item:')
pd.DataFrame(purchases).head(4)

In [ ]:
agg = led.groupby('archetype').agg(demand=('demand','sum'), sold=('sold','sum'),
                                   lost=('lost','sum'), holding=('closing','mean'))
agg['fill_rate'] = agg.sold / agg.demand
print('naive shopkeeper baseline — what SNAPTOCK has to improve on')
print(agg[['fill_rate','holding','lost']].rename(
      columns={'holding':'avg_on_hand','lost':'lost_units'}).round(3).to_string())

tot_l, tot_d = led.lost.sum(), led.demand.sum()
print(f'\noverall fill rate {1 - tot_l/tot_d:.1%}   ({tot_l:,} of {tot_d:,} units lost)')
print(f'demand censoring  {tot_l/tot_d:.1%} of demand never reaches the records.')

### The censoring choice

When stock hits zero the shop records a sale of **zero** — not the demand it could not
serve. A forecaster fit on recorded sales therefore under-predicts, orders less, and
stocks out again.

We forecast on **`sold`**, because that is what a deployed system can actually observe.


In [ ]:
panel = (led.rename(columns={'date':'ds', 'sku_id':'unique_id', 'sold':'y'})
            [['unique_id','ds','y']].copy())
panel['ds'] = pd.to_datetime(panel['ds'])
panel['unique_id'] = panel['unique_id'].astype(str)

cutoff = panel.ds.max() - pd.Timedelta(days=HOLDOUT)
train = panel[panel.ds <= cutoff].copy()
test  = panel[panel.ds >  cutoff].copy()
print(f'panel {panel.shape} | train {train.ds.nunique()}d | test {test.ds.nunique()}d')
print(f'horizon {HORIZON}d = supplier lead time + review period')

## 4 · What kind of demand is this?

Syntetos–Boylan quadrants. ADI = average interval between demands; CV² = squared
coefficient of variation of demand sizes. Cutoffs 1.32 and 0.49. Routing by quadrant is
the classical strategy under test.


In [ ]:
def classify(qtys):
    nonzero = [q for q in qtys if q > 0]
    if len(nonzero) < 2:
        return float('inf'), 0.0, 'slow'
    adi = len(qtys) / len(nonzero)
    mean = float(np.mean(nonzero))
    cv2 = (float(np.std(nonzero)) / mean) ** 2 if mean else 0.0
    if adi < 1.32:
        return adi, cv2, ('smooth' if cv2 < 0.49 else 'erratic')
    return adi, cv2, ('intermittent' if cv2 < 0.49 else 'lumpy')

rows = []
for uid, g in panel.groupby('unique_id'):
    adi, cv2, label = classify(list(g.sort_values('ds').y))
    rows.append({'unique_id': uid, 'adi': min(adi, 30), 'cv2': cv2, 'quadrant': label})
quad = pd.DataFrame(rows).set_index('unique_id')
print(quad.quadrant.value_counts().to_string())

fig, ax = plt.subplots(figsize=(6, 4.2))
for lab, sub in quad.groupby('quadrant'):
    ax.scatter(sub.adi, sub.cv2, s=22, alpha=.75, label=f'{lab} (n={len(sub)})')
ax.axvline(1.32, color='#888', lw=1, ls='--'); ax.axhline(0.49, color='#888', lw=1, ls='--')
ax.set_xscale('log'); ax.set_xlabel('ADI  (avg interval between demands, log)')
ax.set_ylabel('CV²  (variability of demand size)')
ax.set_title('Demand pattern quadrants — 200 warung SKUs')
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

## 5 · Classical baselines

Metrics are **MASE** and **RMSSE**, scaled by in-sample naive error. Not MAPE — it divides
by actual demand, and these series are full of zero-demand days.


In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, AutoETS, CrostonSBA, TSB

scale = {}   # in-sample naive MAE per series — the MASE denominator
for uid, g in train.groupby('unique_id'):
    d = np.abs(np.diff(g.sort_values('ds').y.values))
    scale[uid] = d.mean() if len(d) and d.mean() > 0 else 1.0

sf = StatsForecast(models=[SeasonalNaive(season_length=7), AutoETS(season_length=7),
                           CrostonSBA(), TSB(alpha_d=0.2, alpha_p=0.2)],
                   freq='D', n_jobs=-1)
fc_stats = sf.forecast(df=train, h=HORIZON)
print(list(fc_stats.columns), len(fc_stats), 'rows')
fc_stats.head(3)

## 6 · Chronos-Bolt, zero-shot

No training, no fitting — the context window goes straight to a pretrained model.
It returns **quantiles**, which matters in section 8: safety stock needs a distribution,
and the classical methods give only a point estimate.


In [ ]:
import torch
from chronos import BaseChronosPipeline

# 'cuda:0' is whichever physical GPU you selected above, not necessarily GPU 0
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if torch.cuda.is_available():
    print(f'visible GPUs: {torch.cuda.device_count()}  ->  {torch.cuda.get_device_name(0)}')
    assert torch.cuda.device_count() == 1 or GPU is None, \
        'CUDA_VISIBLE_DEVICES did not take effect — restart the kernel and run the GPU cell first'
pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-base',
                                           device_map=device, torch_dtype=torch.float32)
print('device:', device)

uids = sorted(train.unique_id.unique(), key=int)
ctx = [torch.tensor(train[train.unique_id == u].sort_values('ds').y.values,
                    dtype=torch.float32) for u in uids]

chunks = []
for i in range(0, len(ctx), 64):
    q, _ = pipe.predict_quantiles(ctx[i:i+64], prediction_length=HORIZON,
                                  quantile_levels=[0.1, 0.5, 0.9])
    chunks.append(q)
quantiles = torch.cat(chunks).cpu().numpy()      # (n_series, H, 3)
print('quantiles:', quantiles.shape)

dates = pd.date_range(cutoff + pd.Timedelta(days=1), periods=HORIZON, freq='D')
fc_chr = pd.DataFrame([
    {'unique_id': u, 'ds': d,
     'Chronos': max(0., quantiles[i, h, 1]), 'Chronos_q90': max(0., quantiles[i, h, 2])}
    for i, u in enumerate(uids) for h, d in enumerate(dates)])

## 7 · Forecast accuracy


In [ ]:
METHODS = ['SeasonalNaive','AutoETS','CrostonSBA','TSB','Chronos']
fc = fc_stats.merge(fc_chr, on=['unique_id','ds'], how='outer')
ev = fc.merge(test, on=['unique_id','ds'], how='inner')

def score(df):
    out = {}
    for m in METHODS:
        abs_e = sq_e = 0.; n = 0
        for uid, g in df.groupby('unique_id'):
            s = scale[uid]
            abs_e += (g[m] - g.y).abs().sum() / s
            sq_e  += ((g[m] - g.y) ** 2).sum() / (s ** 2)
            n += len(g)
        out[m] = {'MASE': abs_e / n, 'RMSSE': math.sqrt(sq_e / n)}
    return pd.DataFrame(out).T

overall = score(ev).sort_values('MASE')
print('OVERALL'); print(overall.round(3).to_string())

ev_q = ev.merge(quad[['quadrant']], left_on='unique_id', right_index=True)
by_quad = pd.DataFrame({q: score(g)['MASE'] for q, g in ev_q.groupby('quadrant')})
print('\nMASE BY QUADRANT'); print(by_quad.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
o = overall.sort_values('MASE')
axes[0].barh(range(len(o)), o.MASE, color=[PALETTE[m] for m in o.index], height=.62)
axes[0].set_yticks(range(len(o))); axes[0].set_yticklabels(o.index); axes[0].invert_yaxis()
axes[0].set_xlabel('MASE  (lower is better)'); axes[0].set_title('Overall forecast error')
for i, v in enumerate(o.MASE):
    axes[0].text(v, i, f'  {v:.3f}', va='center', fontsize=8)

cols = [c for c in ['smooth','erratic','intermittent','lumpy'] if c in by_quad]
bq = by_quad.reindex(columns=cols)
x = np.arange(len(cols)); w = .16
for k, m in enumerate(METHODS):
    axes[1].bar(x + k*w, bq.loc[m], w, label=m, color=PALETTE[m])
axes[1].set_xticks(x + 2*w); axes[1].set_xticklabels(cols)
axes[1].set_ylabel('MASE'); axes[1].set_title('By demand quadrant')
axes[1].legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

## 8 · The comparison that matters — inventory outcome

Forecast error is a proxy. What the shop feels is: did I run out, and how much cash is on
the shelf?

Each forecast drives the same reorder rule. Classical methods take safety stock from
residual spread (a normal approximation, standard practice). Chronos uses its own **q90**
directly — no normality assumed.

Better methods sit **up and to the left**: higher fill rate for less stock held.


In [ ]:
Z = 1.28   # ~90% service level
truth = test.set_index(['unique_id','ds']).y

def run_policy(method, use_q90=False):
    served = wanted = held = days = 0.
    for uid, g in fc.groupby('unique_id'):
        g = g.sort_values('ds')
        pred = g[method].clip(lower=0).values
        lead_demand = pred[:LEAD].sum()
        if use_q90:
            safety = max(0., g['Chronos_q90'].values[:LEAD].sum() - lead_demand)
        else:
            safety = Z * train[train.unique_id == uid].y.values.std() * math.sqrt(LEAD)
        reorder = lead_demand + safety
        order_up_to = max(reorder + pred.mean() * 7, reorder + 1)
        on_hand, incoming = order_up_to, collections.Counter()
        for t, d in enumerate(g.ds):
            on_hand += incoming.pop(t, 0)
            want = float(truth.get((uid, d), 0))
            sold = min(want, on_hand); on_hand -= sold
            served += sold; wanted += want; held += on_hand; days += 1
            if on_hand <= reorder and not incoming:
                incoming[t + LEAD] += max(1., order_up_to - on_hand)
    return {'fill_rate': served / max(wanted, 1), 'avg_on_hand': held / max(days, 1)}

results = {m: run_policy(m) for m in METHODS}
results['Chronos (q90)'] = run_policy('Chronos', use_q90=True)
biz = pd.DataFrame(results).T.sort_values('fill_rate', ascending=False)
print(biz.assign(fill_rate=lambda d: (d.fill_rate * 100).round(2)).round(2).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))
for name, r in results.items():
    ax.scatter(r['avg_on_hand'], r['fill_rate'] * 100, s=110,
               color=PALETTE[name.split(' ')[0]], edgecolor='white', linewidth=1.5, zorder=3)
    ax.annotate(name, (r['avg_on_hand'], r['fill_rate'] * 100),
                textcoords='offset points', xytext=(8, 4), fontsize=8)
ax.set_xlabel('average units held  (cash tied up →)')
ax.set_ylabel('fill rate %  (↑ fewer stockouts)')
ax.set_title('Service level vs inventory held — up and left is better')
plt.tight_layout(); plt.show()

## 9 · Reading the result

1. **Does Chronos beat CrostonSBA on MASE?** If not, the prior holds and you ship the
   classical stack — no weights, no GPU, no download inside `docker compose`.
2. **Does the ranking flip in section 8?** If Chronos loses on error but wins on fill rate
   at equal stock, the value is calibrated uncertainty, not point accuracy — and the honest
   conclusion is 'SBA for the point forecast, a distribution for safety stock'.
3. **Does the quadrant table justify routing?** If one method wins everywhere, drop the
   ADI/CV² routing — it is complexity you cannot defend.

### Caveats

- Synthetic data with hand-chosen archetype parameters. Re-run across several `SEED`
  values; if the ranking flips, it depends on assumptions we invented.
- Fit on censored `sold`. Re-run on `demand` to see what the censoring costs — that gap
  is a finding in itself.
- Zero-shot only. Fine-tuning Chronos on synthetic data mostly measures whether it can
  learn our generator.
